In [1]:
import os
import gradio as gr
from roboflow import Roboflow
from PIL import Image

rf = Roboflow(api_key="uBQUwbuRddQGIP6cxHCo")
project = rf.workspace("witeks-workspace").project("dekracoating")
model = project.version(4).model

def scan_drill(image):
    if image is None:
        return {"error": "No image provided"}
    image.save("temp.jpg")
    result = model.predict("temp.jpg", confidence=40).json()
    predictions = []
    for pred in result.get("predictions", []):
        predictions.append({
            "type": pred.get("class"),
            "width_px": round(pred.get("width", 0), 1),
            "height_px": round(pred.get("height", 0), 1),
            "confidence": round(pred.get("confidence", 0), 2)
        })
    return predictions

demo = gr.Interface(
    fn=scan_drill,
    inputs=gr.Image(type="pil", label="Upload a drill photo"),
    outputs=gr.JSON(label="Drill info"),
    title="Drill Scanner – Dekracoating"
)

demo.launch()

/home/codespace/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading Roboflow workspace...
loading Roboflow project...
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import os
import gradio as gr
from roboflow import Roboflow
from PIL import Image

# Black box real dimensions
BLACK_BOX_LENGTH_MM = 300
BLACK_BOX_WIDTH_MM = 190

rf = Roboflow(api_key="uBQUwbuRddQGIP6cxHCo")
project = rf.workspace("witeks-workspace").project("dekracoating")
model = project.version(4).model

def scan_drill(image):
    if image is None:
        return {"error": "No image provided"}

    image.save("temp.jpg")
    result = model.predict("temp.jpg", confidence=40).json()
    predictions = result.get("predictions", [])

    # Debug: print all detected class names so we can find exact black box name
    all_classes = [p.get("class") for p in predictions]
    print("All detected classes:", all_classes)

    # Find black box — check all predictions regardless of case or spacing
    black_box = next(
        (p for p in predictions if "black" in p.get("class", "").lower()),
        None
    )

    if black_box:
        mm_per_pixel_x = BLACK_BOX_LENGTH_MM / black_box["width"]
        mm_per_pixel_y = BLACK_BOX_WIDTH_MM / black_box["height"]
        scale_found = True
        print("Black box found:", black_box.get("class"), "width_px:", black_box["width"])
    else:
        scale_found = False
        print("No black box detected in this photo!")

    output = []
    for pred in predictions:
        class_name = pred.get("class", "")

        if "black" in class_name.lower():
            continue  # skip the black box itself

        entry = {
            "type": class_name,
            "confidence": round(pred.get("confidence", 0), 2),
            "width_px": round(pred.get("width", 0), 1),
            "height_px": round(pred.get("height", 0), 1),
        }

        if scale_found:
            entry["width_mm"] = round(pred.get("width", 0) * mm_per_pixel_x, 1)
            entry["length_mm"] = round(pred.get("height", 0) * mm_per_pixel_y, 1)
            entry["scale_used"] = "black box"
        else:
            entry["width_mm"] = "no black box in photo"
            entry["length_mm"] = "no black box in photo"
            entry["scale_used"] = "none — add black box to photo"

        output.append(entry)

    return output


demo = gr.Interface(
    fn=scan_drill,
    inputs=gr.Image(type="pil", label="Upload a drill photo"),
    outputs=gr.JSON(label="Drill info"),
    title="Drill Scanner – Dekracoating"
)

demo.launch()

loading Roboflow workspace...
loading Roboflow project...
* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


All detected classes: ['black-stand', 'oranje-20', 'blauw-16', 'zwart-12', 'oranje-10', 'oranje-10', 'oranje-10', 'geel-8', 'geel-8', 'geel-8', 'geel-8']
Black box found: black-stand width_px: 692.0
